# 🌍 duckOSM Network Explorer

This notebook allows you to interactively explore and visualize the routing networks generated by `duckOSM`.

In [1]:
import duckdb
import leafmap.foliumap as leafmap
import geopandas as gpd
from shapely import wkt
import os

## 1. Connect to Database
Select your transportation mode (e.g., `driving`, `walking`, `cycling`).

In [2]:
db_path = '../data/output/somerset.duckdb'
mode = 'driving'

con = duckdb.connect(db_path)
con.execute("INSTALL spatial; LOAD spatial;")

# Check available schemas
con.execute("SELECT schema_name FROM information_schema.schemata WHERE schema_name NOT IN ('information_schema', 'pg_catalog', 'main', 'raw')").fetchall()

[('cycling',), ('driving',), ('walking',)]

## 2. Load Network Data
We load the edges into a GeoDataFrame for visualization. Note that we use `ST_AsText(geometry)` because we are now using native DuckDB spatial types.

In [4]:
query = f"""
    SELECT 
        edge_id, osm_id, name, highway, 
        length_m, maxspeed_kmh, cost_s, oneway, from_cell, to_cell,
        ST_AsText(geometry) as geometry
    FROM {mode}.edges
    LIMIT 5000
"""
df = con.execute(query).df()

# Convert WKT to geometry
df['geometry'] = df['geometry'].apply(wkt.loads)
gdf = gpd.GeoDataFrame(df, crs="EPSG:4326")

gdf.head()

,edge_id,osm_id,name,highway,length_m,maxspeed_kmh,cost_s,oneway,from_cell,to_cell,geometry
0,1,16339702,Vermillion Avenue,residential,82.091858,30.0,9.851024,None,645224977386006821,645224977386014593,"LINESTRING (-84.59347 37.08397, -84.59379 37.0..."
1,2,16348221,Monticello Road,secondary,52.389465,60.0,3.143368,None,645224977953839400,645224977953823888,"LINESTRING (-84.62481 37.05165, -84.62461 37.0..."
2,3,16347870,Carl Floyd Road,residential,10.017295,30.0,1.202075,None,644690509583920041,644690509583880587,"LINESTRING (-84.63368 37.08728, -84.63368 37.0..."
3,4,136814036,None,secondary,1624.840454,60.0,97.490433,None,644690510029882067,644690509913567324,"LINESTRING (-84.62159 37.10935, -84.62162 37.1..."
4,5,16340224,None,residential,22.272966,30.0,2.672756,None,644690509557320848,644690509557315276,"LINESTRING (-84.64594 37.08914, -84.64607 37.0..."


## 3. Visualize on Map
Click on any edge to see its properties.

In [5]:
m = leafmap.Map(center=[gdf.geometry.centroid.y.mean(), gdf.geometry.centroid.x.mean()], zoom=13)

# Simple coloring based on highway type
m.add_gdf(
    gdf, 
    layer_name=f"{mode.capitalize()} Network",
    info_mode="on_click"
)
m